# Diagonal integration: [RNA, ATAC]

Diagonal integration combines RNA and ATAC measured in different cells, with no pairing between them. If your RNA and ATAC come from the same cells, as in 10x Multiome, use the vertical tutorial. This tutorial runs iNMF and online_iNMF on `D27mini`, 3,000 RNA and 3,000 ATAC profiles of the benchmark dataset `D27`, then on data in your own format.

Most diagonal methods read ATAC as gene-activity scores, made beforehand with a tool such as Signac or ArchR. GLUE and Seurat_v5 read the peak matrix, and MultiMAP and Seurat_v3 need both. [The folder layout](https://dsichang.github.io/scMultiBench/tutorials/run/#folder-layout) lists each method's ATAC files.

Methods run on Linux. On macOS or Windows, [open this notebook in Colab](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_diagonal_rna_atac.ipynb).

## 1. Install

This cell installs `multibench-sc`. On Colab, choose a T4 GPU runtime first (Runtime -> Change runtime type), because changing it later loses the downloads.

In [ ]:
import numpy, pandas
pins = f"numpy=={numpy.__version__} pandas=={pandas.__version__}"
%pip install -q "multibench-sc>=0.5.0" {pins}

## 2. Download the data and the environments

`mtb.data.fetch` downloads `D27mini` (15 MB) once. `mtb.env.install` downloads the environments for iNMF and online_iNMF. The download is 0.9 GB.

In [ ]:
import multibench as mtb

mtb.data.fetch("D27mini")
METHODS = ["iNMF", "online_iNMF"]
envs = mtb.env.install(METHODS)

## 3. Run the methods

`run` runs each method on `D27mini` and scores its output with the scIB metrics. The clustering metrics measure how well the embedding separates the cell types. The batch metrics measure how well the batches mix. The classification scores measure how well the cell types learned on one batch are predicted in the other batches. `res.summary` has one row per method with its status, run time and scores. Higher is better for every metric.

<details>
<summary>Details</summary>

`params={"Method": {"key": value}}` sets a method's parameters, and `mtb.params_for` lists them. Many methods take none.

`mtb.load_run("out/D27mini")` reloads these results later without running anything.

</details>

In [ ]:
res = mtb.run("D27mini", "diagonal", methods=METHODS,
                  out_dir="out/D27mini")
res.summary

## 4. Plot

`res.plot()` draws the scores as a bubble table. The lightest fill is the lowest value in this figure, not zero.

<details>
<summary>Details</summary>

Metrics are grouped by family: blue for clustering, green for batch correction, pink for classification. Each family starts with an `Overall` bar. Its length and colour both show the family score.

A column whose rows all hold the same value is drawn grey, and a warning names it. A figure of one method is grey everywhere.

</details>

In [ ]:
res.plot()

## 5. Your own data

Your data needs raw RNA counts in one `.h5ad` file and ATAC in another, with a cell type for each cell in both. iNMF and online_iNMF read ATAC as gene-activity scores, and online_iNMF needs about 5,000 cells in total. Gene-activity scores are an ATAC matrix with one column per gene, as Signac or ArchR compute it.

Starting from something else? See the recipes for [labels in a CSV file](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-labels-csv) and [no labels](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-no-labels).

<details>
<summary>Details: export</summary>

The names in quotes say where your file holds each part. `"X"` is the main matrix, `"obsm:atac"` the matrix `obsm["atac"]`, `"layer:counts"` the layer `layers["counts"]`, and `"obs:celltype"` the column `obs["celltype"]`, which `"celltype"` alone names too. A wrong name after `obsm:`, `layer:` or `obs:` raises an error that lists the names the file has.

`labels=` needs the column in both files. With cell types for the RNA cells only, leave `labels=` out and follow the recipe for no labels.

For ATAC as a peak matrix, pass `atac_kind="peak"`. GLUE and Seurat_v5 read peaks. [The folder layout](https://dsichang.github.io/scMultiBench/tutorials/run/#folder-layout) lists the files each method needs.

Without `methods=`, `scan` lists every method that reads the modalities in the folder.

</details>

The cells below read your data, write it as a dataset folder, check the folder and run the same methods on it. So that the notebook runs as it is, the next cell writes two small example files, `my_rna.h5ad` and `my_atac.h5ad`, from 90% of the RNA profiles and 90% of the ATAC profiles of `D27mini`.

In [ ]:
mtb.data.write_example("D27mini")

Read your files. For your own data, change every file name marked `# <-`. On Colab, upload the files first: click the folder icon on the left, then the upload icon.

In [ ]:
import anndata as ad

rna = ad.read_h5ad("my_rna.h5ad")     # <- your RNA file
atac = ad.read_h5ad("my_atac.h5ad")   # <- your ATAC file, other cells
print(rna)
print(atac)

Write the dataset folder. Change each name in quotes marked `# <-`: give `NAME` a new value, and write the names that your files use for the other parts. The printout above lists the names in your files. Write a name listed after `obs:` with that prefix, as `"obs:celltype"`.

Compare the cell counts above the table with your data.

In [ ]:
NAME = "MYDIAG_RNA_ATAC"   # <- a new name for your own data
mtb.io.export_dataset(rna, "mydata/" + NAME,
                      rna="X",                    # <- where the RNA counts are:
                                                  #    "X" or "layer:counts"
                      atac=atac,                  # the ATAC file, values in X
                      atac_kind="gene_activity",  # <- or "peak"
                      labels="obs:celltype",      # <- the column in both files
                      category="diagonal", overwrite=True)
mtb.scan("mydata/" + NAME, "diagonal", methods=METHODS)

Run the same methods on the folder:

In [ ]:
mine = mtb.run("mydata/" + NAME, "diagonal", methods=METHODS,
                   out_dir="out/" + NAME)
mine.summary

In [ ]:
mine.plot()

## 6. Stored results

The package ships stored results for 11 methods on `D28`. `load_results` reads them and `mtb.plot.bubble` draws them, without running anything.

<details>
<summary>Details</summary>

There are no stored results for `D27mini`. `D28` is another benchmark dataset of the same task.

The stored results are the package's own runs of the methods.

Scores of your runs differ a little from the stored ones: most methods give a slightly different embedding on each run.

</details>

In [ ]:
long = mtb.load_results("diagonal", dataset="D28")
mtb.plot.bubble(long)

## 7. Every method of this task

14 methods have a variant for this task. [This notebook](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_diagonal_rna_atac_all.ipynb) runs them all. It downloads 8 envs, 13.9 GB on a CPU host, 13.2 GB on a GPU host.

## Troubleshooting

`res.summary[res.summary.status != "scored"]` lists each method that failed or was skipped. Its `reason` column ends with the method's error output.

<details>
<summary>Details</summary>

| symptom | fix |
|---|---|
| a method fails on a small dataset | `mtb.scan` names what the method needs from the data's size in its `caveat` column |
| a warning that values are not whole numbers | export raw counts, for example with `rna="layer:counts"` |
| `... matrix/data as cells x features` | the matrix is transposed: export it again with `mtb.io.export_dataset` |
| a method times out | raise `timeout=` in `run` |
| batch metrics use the wrong batches | `res.rescore(batch=my_vector)` scores again without running the methods |

</details>

## Next steps

- The guides: [run](https://dsichang.github.io/scMultiBench/tutorials/run/), [evaluate](https://dsichang.github.io/scMultiBench/tutorials/evaluate/), [plot](https://dsichang.github.io/scMultiBench/tutorials/plot/) and [discover methods](https://dsichang.github.io/scMultiBench/tutorials/discover/).
- The [interactive explorer](https://shiny.maths.usyd.edu.au/scMultiBench/) has the full benchmark's rankings, with no install needed.